# BT4221 Group Project — Agent Pipeline

A fully autonomous multi-agent LangGraph pipeline for Steam Reviews classification.

Each **domain agent** reasons freely from raw statistics with no guardrails — it never sees the executable schema. A separate **validator agent** (also LLM-backed) then translates the free-form decisions into an executable format. PySpark executes the validated output.

This gives us three clean layers:
- **Domain agent** — *what should be done and why* (no schema shown)
- **Validator agent** — *how to express that in executable form* (knows the schema, translates)
- **PySpark executor** — runs the validated instructions

State stores both `_raw` and `_validated` versions of every agent decision for full traceability.

| Node | Role |
|------|------|
| 1 | Load raw data + non-agent preparation |
| 2 | Compute raw summary statistics |
| 3 | **Cleaning Agent** — reasons freely from raw stats |
| 4 | **Cleaning Validator** — translates to executable schema |
| 5 | PySpark executes validated cleaning |
| 6 | Compute detailed post-cleaning statistics |
| 7 | **FE Agent** — reasons freely from cleaned stats |
| 8 | **FE Validator** — translates to executable schema |
| 9 | PySpark dynamic dispatcher executes transformations |
| 10 | Compute post-FE summary statistics |
| 11 | **Model Config Agent** — reasons freely |
| 12 | **Model Config Validator** — translates to executable schema |
| 13 | PySpark trains models using validated configs |
| 14 | **Evaluation Agent** — domain-grounded analysis (no validator needed) |
| 15 | Export all outputs |

## 1. Install & Import Dependencies

In [1]:
!pip install langgraph openai kagglehub pyspark python-dotenv -q

In [3]:
import os, json, glob, shutil
import numpy as np
import pandas as pd
import kagglehub
from openai import OpenAI

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import (
    VectorAssembler, Tokenizer, StopWordsRemover, HashingTF, IDF, Imputer
)
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator, MulticlassClassificationEvaluator
)
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional, List

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI()
print("All imports successful.")

All imports successful.


In [ ]:
initial_state: AgentState = {
    "df":                           None,
    "train_df":                     None,
    "valid_df":                     None,
    "raw_summary":                  None,
    "cleaned_summary":              None,
    "fe_summary":                   None,
    "cleaning_decisions_raw":       None,
    "cleaning_decisions_validated": None,
    "fe_decisions_raw":             None,
    "fe_decisions_validated":       None,
    "model_config_raw":             None,
    "model_config_validated":       None,
    "evaluation_decisions":         None,
    "dense_feature_cols":           None,
    "skipped_instructions":         None,
    "model_1_results":              None,
    "model_2_results":              None,
    "model_1_best":                 None,
    "model_2_best":                 None,
    "feature_importance":           None,
    "output_json_path":             None,
    "evaluation_report_path":       None,
}

final_state = pipeline.invoke(initial_state)

cfg     = final_state["model_config_validated"]
m1_name = cfg.get("model_1", {}).get("pyspark_class", "Model1")
m2_name = cfg.get("model_2", {}).get("pyspark_class", "Model2")
pri_met = cfg.get("primary_metric", "areaUnderROC")

print("\n" + "="*60)
print("PIPELINE COMPLETE")
print("="*60)
print(f"Agent decisions:   {final_state['output_json_path']}")
print(f"Evaluation report: {final_state['evaluation_report_path']}")
print(f"{m1_name}: {pri_met}={final_state['model_1_best'].get(pri_met) if final_state['model_1_best'] else 'N/A'}")
print(f"{m2_name}: {pri_met}={final_state['model_2_best'].get(pri_met) if final_state['model_2_best'] else 'N/A'}")
print(f"Selected model: {final_state['evaluation_decisions'].get('selected_model')}")


NODE 1: LOAD & NON-AGENT PREPARATION
Dataset path: /Users/joshlim/.cache/kagglehub/datasets/najzeko/steam-reviews-2021/versions/1


Rows after non-agent preparation: 9,635,437
Columns (22): ['app_id', 'app_name', 'review_id', 'language', 'review', 'timestamp_created', 'timestamp_updated', 'recommended', 'votes_helpful', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'author_steamid', 'author_num_games_owned', 'author_num_reviews', 'author_playtime_forever', 'author_playtime_last_two_weeks', 'author_playtime_at_review', 'author_last_played']

NODE 2: RAW SUMMARY STATISTICS


Rows: 9,635,437 | Columns: 22
Class distribution: {1: 8584444, 0: 1050993}
Duplicates — exact: 54,769 | review_id: 54,769 | user+app: 55,027
Columns with nulls: ['review', 'author_playtime_forever', 'author_playtime_last_two_weeks', 'author_playtime_at_review', 'author_last_played']

NODE 3: CLEANING AGENT

CLEANING AGENT — RAW DECISIONS
The dataset contains a significant number of rows and various columns that may require cleaning to ensure the quality and reliability of the machine learning model. Observations such as missing values, duplicates, class imbalance, and outlier presence indicate the need for specific cleaning steps. Each decision is based on the statistics provided, aiming to enhance data integrity and model performance.

--- RAW DECISIONS ---
{
  "missing_values": {
    "observations": "The 'review' column has 16,750 missing values, which is about 1.73% of the total rows. Additionally, 'author_playtime_at_review' has 11,847 missing values (0.12%).",
    "action": "Remov

  [dup] Dropped exact duplicates
  [dup] Kept latest review per user+app
  [drop] Columns dropped: ['app_id', 'review_id']
  [null] drop_rows for review


  [null] impute_median=1888.0000 for author_playtime_at_review
  [outlier] dropped rows where votes_helpful > 99
  [outlier] dropped rows where author_num_games_owned > 99


...

    {'numTrees': 100, 'maxDepth': 10, 'minInstancesPerNode': 2, 'featureSubsetStrategy': 'sqrt'} -> areaUnderROC=0.7365


    {'numTrees': 200, 'maxDepth': 15, 'minInstancesPerNode': 2, 'featureSubsetStrategy': 'sqrt'} -> areaUnderROC=0.7375


    {'numTrees': 100, 'maxDepth': 10, 'minInstancesPerNode': 5, 'featureSubsetStrategy': 'log2'} -> areaUnderROC=0.7365


    {'numTrees': 150, 'maxDepth': 10, 'minInstancesPerNode': 10, 'featureSubsetStrategy': 'sqrt'} -> areaUnderROC=0.7362


    {'numTrees': 300, 'maxDepth': 5, 'minInstancesPerNode': 2, 'featureSubsetStrategy': 'sqrt'} -> areaUnderROC=0.7261

Best LogisticRegression: areaUnderROC=0.6833
Best RandomForestClassifier: areaUnderROC=0.7375
Overlapping top-10: ['is_early_access_review', 'review_length_chars', 'review_length_words', 'review_created_datetime', 'punctuation_count', 'exclamation_count', 'question_count']

NODE 14: EVALUATION AGENT

EVALUATION AGENT
The evaluation process involved comparing model performance metrics to determine the best-performing model. Feature importance was analyzed to understand the driving factors behind review sentiment, focusing on the implications of each feature in the context of Steam. The insights gained were then synthesized into practical recommendations for developers and the platform, while also acknowledging the limitations of the analysis.

--- EVALUATION SUMMARY ---
  Selected: RandomForestClassifier (areaUnderROC=0.7375)
  Runner-up: LogisticRegression (areaUnderR

## 2. Define AgentState

All agent decisions — both raw (what the domain agent decided) and validated (what the validator translated it into) — are stored in state for full traceability.

In [49]:
class AgentState(TypedDict):
    # Data
    df:                          object
    train_df:                    object
    valid_df:                    object

    # Summary statistics (inputs to agents)
    raw_summary:                 Optional[dict]
    cleaned_summary:             Optional[dict]
    fe_summary:                  Optional[dict]

    # Cleaning agent
    cleaning_decisions_raw:      Optional[dict]  # What the Cleaning Agent decided
    cleaning_decisions_validated: Optional[dict] # What the Cleaning Validator translated

    # FE agent
    fe_decisions_raw:            Optional[dict]  # What the FE Agent decided
    fe_decisions_validated:      Optional[dict]  # What the FE Validator translated

    # Model config agent
    model_config_raw:            Optional[dict]  # What the Model Config Agent decided
    model_config_validated:      Optional[dict]  # What the Model Config Validator translated

    # Evaluation agent (no validator needed — output is narrative, not executed)
    evaluation_decisions:        Optional[dict]

    # FE outputs
    dense_feature_cols:          Optional[List[str]]
    skipped_instructions:        Optional[List[dict]]

    # Model results
    model_1_results:             Optional[list]
    model_2_results:             Optional[list]
    model_1_best:                Optional[dict]
    model_2_best:                Optional[dict]
    feature_importance:          Optional[dict]

    # Exports
    output_json_path:            Optional[str]
    evaluation_report_path:      Optional[str]

print("AgentState defined.")
print("Fields:", list(AgentState.__annotations__.keys()))

AgentState defined.
Fields: ['df', 'train_df', 'valid_df', 'raw_summary', 'cleaned_summary', 'fe_summary', 'cleaning_decisions_raw', 'cleaning_decisions_validated', 'fe_decisions_raw', 'fe_decisions_validated', 'model_config_raw', 'model_config_validated', 'evaluation_decisions', 'dense_feature_cols', 'skipped_instructions', 'model_1_results', 'model_2_results', 'model_1_best', 'model_2_best', 'feature_importance', 'output_json_path', 'evaluation_report_path']


## 3. Build SparkSession

In [50]:
spark = SparkSession.builder \
    .appName("BT4221_AgentPipeline") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.1.1


## 4. Helper: Call Agent

In [51]:
def call_agent(prompt: str, label: str) -> dict:
    """Call GPT-4o-mini, parse JSON, print reasoning clearly."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    result = json.loads(response.choices[0].message.content)
    print(f"\n{'='*60}")
    print(f"{label}")
    print('='*60)
    reasoning = result.get("reasoning", result.get("translation_notes", "(no reasoning provided)"))
    print(reasoning)
    return result

## 5. Pipeline Nodes

### Node 1: Load & Non-Agent Preparation

Fixed, deterministic steps that require no agent input: download from Kaggle, drop index, rename columns, standardise types, filter English rows.

In [52]:
def node_load_data(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 1: LOAD & NON-AGENT PREPARATION")
    print("="*60)

    path = kagglehub.dataset_download("najzeko/steam-reviews-2021")
    print(f"Dataset path: {path}")

    df = spark.read.csv(
        os.path.join(path, "steam_reviews.csv"),
        header=True, inferSchema=False, multiLine=True, escape='"'
    )
    df = df.drop("_c0")

    for c in df.columns:
        if "author." in c:
            df = df.withColumnRenamed(c, c.replace("author.", "author_"))

    bool_cols = ["recommended", "steam_purchase", "received_for_free", "written_during_early_access"]
    for c in bool_cols:
        df = df.withColumn(c, F.when(F.col(c) == "True", 1).otherwise(0).cast("int"))

    num_cols = [
        "votes_helpful", "votes_funny", "weighted_vote_score", "comment_count",
        "author_num_games_owned", "author_num_reviews", "author_playtime_forever",
        "author_playtime_last_two_weeks", "author_playtime_at_review", "author_last_played"
    ]
    for c in num_cols:
        df = df.withColumn(c, F.expr(f"try_cast(`{c}` AS DOUBLE)"))
    for c in ["timestamp_created", "timestamp_updated"]:
        df = df.withColumn(c, F.expr(f"try_cast(try_cast(`{c}` AS DOUBLE) AS BIGINT)"))

    df = df.withColumn("language", F.trim(F.lower(F.col("language"))))
    df = df.withColumn("app_name", F.trim(F.col("app_name")))
    df = df.filter(F.col("language") == "english")

    total = df.count()
    print(f"Rows after non-agent preparation: {total:,}")
    print(f"Columns ({len(df.columns)}): {df.columns}")

    state["df"] = df
    return state

### Node 2: Compute Raw Summary Statistics

Provides the Cleaning Agent with everything it needs to reason: null counts, duplicate counts, numeric distributions, class balance, text stats.

In [53]:
def node_raw_summary(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 2: RAW SUMMARY STATISTICS")
    print("="*60)

    df    = state["df"]
    total = df.count()
    dtype_map = dict(df.dtypes)
    double_cols = [c for c, t in df.dtypes if t == "double"]

    # Null counts
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
    ]).collect()[0]

    # Numeric stats
    numeric_stats = {}
    for c in double_cols:
        row = df.select(
            F.mean(c).alias("mean"), F.stddev(c).alias("std"),
            F.min(c).alias("min"),  F.max(c).alias("max"),
            F.skewness(c).alias("skew"), F.kurtosis(c).alias("kurt"),
            F.percentile_approx(c, 0.5).alias("median"),
            F.percentile_approx(c, 0.95).alias("p95"),
            F.percentile_approx(c, 0.99).alias("p99"),
        ).collect()[0]
        numeric_stats[c] = {k: (round(float(row[k]), 4) if row[k] is not None else None) for k in row.asDict()}

    # Duplicate counts
    exact_dupes      = total - df.dropDuplicates().count()
    dupe_review_ids  = total - df.dropDuplicates(["review_id"]).count() if "review_id" in df.columns else 0
    dupe_user_app    = total - df.dropDuplicates(["author_steamid", "app_id"]).count()

    # Class distribution
    class_dist = {int(r["recommended"]): int(r["count"])
                  for r in df.groupBy("recommended").count().collect()}

    # Text stats
    text_row = df.select(
        F.mean(F.length("review")).alias("mean_chars"),
        F.stddev(F.length("review")).alias("std_chars"),
        F.min(F.length("review")).alias("min_chars"),
        F.max(F.length("review")).alias("max_chars"),
        F.percentile_approx(F.length("review"), 0.5).alias("median_chars"),
        F.percentile_approx(F.length("review"), 0.95).alias("p95_chars"),
    ).collect()[0]
    text_stats = {k: (round(float(text_row[k]), 2) if text_row[k] is not None else None) for k in text_row.asDict()}

    raw_summary = {
        "total_rows": int(total),
        "columns": df.columns,
        "dtype_map": dtype_map,
        "null_counts": {c: int(null_counts[c]) for c in df.columns},
        "null_pct": {c: round(int(null_counts[c]) / total * 100, 4) for c in df.columns},
        "duplicate_counts": {
            "exact_duplicates": int(exact_dupes),
            "duplicate_review_ids": int(dupe_review_ids),
            "duplicate_user_app_pairs": int(dupe_user_app),
        },
        "class_distribution": class_dist,
        "numeric_stats": numeric_stats,
        "review_text_stats": text_stats,
        "top_app_names": [r["app_name"] for r in df.groupBy("app_name").count().orderBy(F.desc("count")).limit(10).collect()],
    }

    print(f"Rows: {total:,} | Columns: {len(df.columns)}")
    print(f"Class distribution: {class_dist}")
    print(f"Duplicates — exact: {exact_dupes:,} | review_id: {dupe_review_ids:,} | user+app: {dupe_user_app:,}")
    print(f"Columns with nulls: {[c for c in df.columns if null_counts[c] > 0]}")

    state["raw_summary"] = raw_summary
    return state

### Node 3: Cleaning Agent

Receives only raw statistics. No schema shown. Decides all cleaning steps freely.

In [54]:
def node_cleaning_agent(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 3: CLEANING AGENT")
    print("="*60)

    prompt = f"""You are a data cleaning expert analysing a dataset for a machine learning pipeline.
The task is binary classification: predict whether a Steam game review is positive (recommended=1) or negative (recommended=0).

You have been given raw summary statistics of the dataset after basic preparation (column renaming, type casting, English filtering only).
Reason from these statistics and decide ALL cleaning steps that should be applied.

Do NOT assume any fixed set of options — decide what makes sense based on what you observe.
Consider: duplicate handling, missing value treatment, outlier handling, column removal, class balancing, and sampling.

Raw dataset statistics:
{json.dumps(state['raw_summary'], indent=2)}

Describe each cleaning decision in plain English. For each decision, explain:
- What you observed in the statistics that led to this decision
- What action should be taken
- Why this is the right approach

Return a JSON object with your decisions. Use whatever structure makes sense to you.
The only requirement is that your response includes a 'reasoning' field with your overall reasoning,
and a 'decisions' field containing your cleaning decisions in whatever format you prefer.

{{
  "reasoning": "Your overall reasoning process...",
  "decisions": {{
    ... your decisions in whatever structure makes sense ...
  }}
}}"""

    result = call_agent(prompt, "CLEANING AGENT — RAW DECISIONS")
    print("\n--- RAW DECISIONS ---")
    print(json.dumps(result.get("decisions", {}), indent=2))

    state["cleaning_decisions_raw"] = result
    return state

### Node 4: Cleaning Validator

Receives the Cleaning Agent's free-form decisions. Translates them into an executable schema that PySpark can act on. The validator knows the schema — the domain agent does not.

In [55]:
def node_cleaning_validator(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 4: CLEANING VALIDATOR")
    print("="*60)

    prompt = f"""You are a technical translator for a PySpark data pipeline.
A data cleaning agent has produced free-form cleaning decisions.
Your job is to translate those decisions into a precise executable schema that PySpark code can act on.

The cleaning agent's decisions:
{json.dumps(state['cleaning_decisions_raw'], indent=2)}

Translate each decision into the following executable schema.
If a decision cannot be mapped to any executable action, include it in 'unresolvable' with an explanation.

Return ONLY a valid JSON object with no markdown:
{{
  "duplicate_handling": {{
    "exact_duplicates":       {{"action": "drop" | "skip"}},
    "duplicate_review_ids":   {{"action": "drop" | "skip"}},
    "duplicate_user_app":     {{"action": "keep_latest" | "drop" | "skip"}}
  }},
  "columns_to_drop": ["col1", "col2"],
  "missing_value_handling": {{
    "column_name": {{"action": "drop_rows" | "impute_mean" | "impute_median" | "impute_zero" | "impute_mode" | "skip"}}
  }},
  "outlier_handling": [
    {{"column": "col", "action": "cap_percentile" | "drop_rows" | "skip", "threshold": 99}}
  ],
  "sampling": {{
    "method": "stratified_balanced" | "stratified_proportional",
    "seed": 42
  }},
  "unresolvable": [
    {{"original_decision": "...", "reason": "why it cannot be executed"}}
  ],
  "translation_notes": "Explain how you mapped each raw decision to the schema."
}}"""

    result = call_agent(prompt, "CLEANING VALIDATOR — TRANSLATED SCHEMA")
    print("\n--- VALIDATED SCHEMA ---")
    print(json.dumps({k: v for k, v in result.items() if k != "translation_notes"}, indent=2))
    if result.get("unresolvable"):
        print(f"\n  Unresolvable decisions: {len(result['unresolvable'])}")

    state["cleaning_decisions_validated"] = result
    return state

### Node 5: Apply Cleaning

PySpark executes the validated cleaning schema.

In [56]:
def node_apply_cleaning(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 5: APPLY CLEANING")
    print("="*60)

    df = state["df"]
    d  = state["cleaning_decisions_validated"]
    before = df.count()

    # Duplicate handling
    dup = d.get("duplicate_handling", {})
    if dup.get("exact_duplicates", {}).get("action") == "drop":
        df = df.dropDuplicates()
        print("  [dup] Dropped exact duplicates")
    if dup.get("duplicate_review_ids", {}).get("action") == "drop" and "review_id" in df.columns:
        df = df.dropDuplicates(["review_id"])
        print("  [dup] Dropped duplicate review_ids")
    ua_action = dup.get("duplicate_user_app", {}).get("action", "skip")
    if ua_action == "keep_latest" and "author_steamid" in df.columns:
        w = Window.partitionBy("author_steamid", "app_id").orderBy(F.col("timestamp_updated").desc())
        df = df.withColumn("_rank", F.rank().over(w)).filter(F.col("_rank") == 1).drop("_rank")
        print("  [dup] Kept latest review per user+app")
    elif ua_action == "drop" and "author_steamid" in df.columns:
        df = df.dropDuplicates(["author_steamid", "app_id"])
        print("  [dup] Dropped duplicate user+app pairs")

    # Drop columns
    protected = {"review", "app_name", "recommended"}
    to_drop = [c for c in d.get("columns_to_drop", []) if c in df.columns and c not in protected]
    if to_drop:
        df = df.drop(*to_drop)
        print(f"  [drop] Columns dropped: {to_drop}")

    # Missing value handling
    for col_name, spec in d.get("missing_value_handling", {}).items():
        if col_name not in df.columns:
            continue
        action = spec.get("action", "skip")
        dtype  = dict(df.dtypes).get(col_name, "string")
        if action == "drop_rows":
            df = df.dropna(subset=[col_name])
            print(f"  [null] drop_rows for {col_name}")
        elif action == "impute_mean" and dtype == "double":
            val = df.select(F.mean(col_name)).collect()[0][0]
            if val is not None:
                df = df.fillna({col_name: float(val)})
                print(f"  [null] impute_mean={val:.4f} for {col_name}")
        elif action == "impute_median" and dtype == "double":
            val = df.approxQuantile(col_name, [0.5], 0.01)[0]
            df = df.fillna({col_name: float(val)})
            print(f"  [null] impute_median={val:.4f} for {col_name}")
        elif action == "impute_zero":
            df = df.fillna({col_name: 0})
            print(f"  [null] impute_zero for {col_name}")
        elif action == "impute_mode":
            mode_val = df.groupBy(col_name).count().orderBy(F.desc("count")).first()[col_name]
            df = df.fillna({col_name: mode_val})
            print(f"  [null] impute_mode={mode_val} for {col_name}")

    # Outlier handling
    for spec in d.get("outlier_handling", []):
        col_name = spec.get("column")
        action   = spec.get("action", "skip")
        if col_name not in df.columns or action == "skip":
            continue
        if action == "cap_percentile":
            pct = spec.get("threshold", 99) / 100.0
            cap = df.approxQuantile(col_name, [pct], 0.01)[0]
            df  = df.withColumn(col_name, F.when(F.col(col_name) > cap, cap).otherwise(F.col(col_name)))
            print(f"  [outlier] capped {col_name} at p{spec.get('threshold')} ({cap:.2f})")
        elif action == "drop_rows":
            thr = spec.get("threshold")
            if thr:
                df = df.filter(F.col(col_name) <= thr)
                print(f"  [outlier] dropped rows where {col_name} > {thr}")

    # Sampling
    samp = d.get("sampling", {})
    method = samp.get("method", "stratified_balanced")
    seed   = samp.get("seed", 42)
    class_counts = df.groupBy("recommended").count().collect()
    count_dict   = {int(r["recommended"]): int(r["count"]) for r in class_counts}
    minority     = min(count_dict.values())
    if method == "stratified_balanced":
        fractions = {cls: min((minority / cnt), 1.0) for cls, cnt in count_dict.items()}
    else:
        total_n = sum(count_dict.values())
        target  = minority * 2 // 3
        fractions = {cls: min(target / total_n, 1.0) for cls in count_dict}
    df = df.sampleBy("recommended", fractions=fractions, seed=seed)

    after = df.count()
    print(f"\n  Rows: {before:,} -> {after:,}")
    print(f"  Sampling ({method}): {fractions}")

    state["df"] = df
    return state

### Node 6: Compute Detailed Post-Cleaning Statistics

Deep per-feature statistics fed to the FE Agent: distributions, skewness, kurtosis, class-conditional means, correlations, text analysis.

In [57]:
def node_cleaned_summary(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 6: DETAILED POST-CLEANING STATISTICS")
    print("="*60)

    df    = state["df"]
    total = df.count()
    dtype_map    = dict(df.dtypes)
    double_cols  = [c for c, t in df.dtypes if t == "double"]
    int_cols     = [c for c, t in df.dtypes if t in ("int", "integer")]

    # Deep numeric stats
    numeric_detail = {}
    for c in double_cols + int_cols:
        row = df.select(
            F.mean(c).alias("mean"),   F.stddev(c).alias("std"),
            F.min(c).alias("min"),    F.max(c).alias("max"),
            F.skewness(c).alias("skew"), F.kurtosis(c).alias("kurt"),
            F.percentile_approx(c, 0.25).alias("p25"),
            F.percentile_approx(c, 0.5).alias("p50"),
            F.percentile_approx(c, 0.75).alias("p75"),
            F.percentile_approx(c, 0.95).alias("p95"),
            F.percentile_approx(c, 0.99).alias("p99"),
            F.count(F.when(F.col(c) == 0, 1)).alias("zero_count"),
            F.count(F.when(F.col(c).isNull(), 1)).alias("null_count"),
        ).collect()[0]
        numeric_detail[c] = {
            k: (round(float(row[k]), 4) if row[k] is not None and k not in ("zero_count", "null_count")
                else (int(row[k]) if row[k] is not None else None))
            for k in row.asDict()
        }

    # Class-conditional means
    class_conditional = {}
    for c in double_cols:
        rows = df.groupBy("recommended").agg(F.mean(c).alias("mean")).collect()
        class_conditional[c] = {
            int(r["recommended"]): round(float(r["mean"]), 4)
            for r in rows if r["mean"] is not None
        }

    # Correlation matrix
    corr_pairs = []
    if len(double_cols) >= 2:
        asm = VectorAssembler(inputCols=double_cols, outputCol="_v", handleInvalid="skip")
        arr = Correlation.corr(asm.transform(df.select(double_cols).dropna()), "_v").collect()[0][0].toArray()
        for i in range(len(double_cols)):
            for j in range(i+1, len(double_cols)):
                r = float(arr[i][j])
                if abs(r) > 0.3:
                    corr_pairs.append({"col_a": double_cols[i], "col_b": double_cols[j], "r": round(r, 4)})
        corr_pairs.sort(key=lambda x: abs(x["r"]), reverse=True)

    # Text stats
    text_detail = {}
    if "review" in df.columns:
        row = df.select(
            F.mean(F.length("review")).alias("mean_chars"),
            F.stddev(F.length("review")).alias("std_chars"),
            F.percentile_approx(F.length("review"), 0.5).alias("median_chars"),
            F.percentile_approx(F.length("review"), 0.95).alias("p95_chars"),
            F.mean(F.size(F.split(F.trim(F.col("review")), r"\s+"))).alias("mean_words"),
            F.percentile_approx(F.size(F.split(F.trim(F.col("review")), r"\s+")), 0.5).alias("median_words"),
            F.percentile_approx(F.size(F.split(F.trim(F.col("review")), r"\s+")), 0.95).alias("p95_words"),
            F.count(F.when(F.length("review") < 20, 1)).alias("very_short_count"),
            F.count(F.when(F.length("review") > 5000, 1)).alias("very_long_count"),
        ).collect()[0]
        text_detail = {k: (round(float(row[k]), 2) if row[k] is not None and k not in ("very_short_count","very_long_count") else (int(row[k]) if row[k] is not None else None)) for k in row.asDict()}

        cl_rows = df.groupBy("recommended").agg(
            F.mean(F.length("review")).alias("mean_chars"),
            F.mean(F.size(F.split(F.trim(F.col("review")), r"\s+"))).alias("mean_words"),
        ).collect()
        text_detail["class_conditional"] = {
            int(r["recommended"]): {"mean_chars": round(float(r["mean_chars"]),2), "mean_words": round(float(r["mean_words"]),2)}
            for r in cl_rows if r["mean_chars"] is not None
        }

    class_dist = {int(r["recommended"]): int(r["count"]) for r in df.groupBy("recommended").count().collect()}
    sample_reviews = [r["review"][:300] for r in df.select("review").limit(3).collect()]

    cleaned_summary = {
        "total_rows": int(total),
        "columns": df.columns,
        "dtype_map": dtype_map,
        "class_distribution": class_dist,
        "numeric_detail": numeric_detail,
        "class_conditional_means": class_conditional,
        "correlation_pairs_above_0_3": corr_pairs[:30],
        "text_statistics": text_detail,
        "sample_reviews": sample_reviews,
        "timestamp_columns": [c for c in df.columns if "timestamp" in c or "last_played" in c],
        "task_description": "Binary classification: predict recommended (1=positive, 0=negative) from Steam review metadata and text.",
        "domain_context": "Steam is a PC gaming platform. Reviews are written by players. Metadata includes playtime, voting, purchase info, and timestamps."
    }

    print(f"Rows: {total:,} | Columns: {len(df.columns)}")
    print(f"Class distribution: {class_dist}")
    print(f"Numeric columns: {double_cols}")
    print(f"Correlation pairs (|r|>0.3): {len(corr_pairs)}")

    state["cleaned_summary"] = cleaned_summary
    return state

### Node 7: Feature Engineering Agent

Receives deep per-feature statistics. No transformation types or schema shown. Decides all feature engineering from scratch.

In [58]:
def node_fe_agent(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 7: FEATURE ENGINEERING AGENT")
    print("="*60)

    prompt = f"""You are a feature engineering expert analysing a cleaned dataset for a machine learning pipeline.
The task is binary classification: predict whether a Steam game review is positive (recommended=1) or negative (recommended=0).

You have been given detailed statistics of the cleaned dataset.
Your job is to decide ALL feature engineering transformations from scratch, based purely on what you observe in the statistics.

Do NOT assume any standard set of transformations — reason from what you see.
Consider the numeric distributions, skewness, correlations, class-conditional differences, text characteristics, and timestamps.

Steam domain context: reviews have metadata (playtime, votes, purchase info) and free text. Players write reviews after playing.

IMPORTANT: The column called "review" contains raw free text written by players. It CANNOT be used directly as a model feature.
You must extract numerical features from it (e.g. text length, punctuation counts, TF-IDF vectors) before it can be used.
After all text features have been extracted, you should decide whether to drop the raw "review" column from the dataset,
since it serves no further purpose and would cause errors if passed directly to the model assembler.

Cleaned dataset statistics:
{json.dumps(state['cleaned_summary'], indent=2)}

For each transformation you decide on, describe:
- What column(s) it operates on
- What the output column should be called
- What mathematical or logical operation it performs
- Why you chose this based on what you observed in the statistics

You may invent any transformation you think is appropriate.
Return your decisions in whatever JSON structure makes sense to you.
The only requirement is a 'reasoning' field with your overall reasoning and a 'transformations' field listing your decisions.

{{
  "reasoning": "Your reasoning process...",
  "transformations": [
    {{
      "description": "plain English description of what this transformation does",
      "input_columns": ["col1"],
      "output_column": "new_col_name",
      "operation": "describe the mathematical/logical operation",
      "justification": "why this is useful based on the statistics"
    }}
  ]
}}"""

    result = call_agent(prompt, "FE AGENT — RAW DECISIONS")
    print(f"\n  Transformations decided: {len(result.get('transformations', []))}")
    for i, t in enumerate(result.get("transformations", []), 1):
        print(f"  {i:2d}. {t.get('output_column','?')} — {t.get('description','')[:80]}")

    state["fe_decisions_raw"] = result
    return state

### Node 8: FE Validator

Receives the FE Agent's free-form transformation descriptions. Translates each into a precise executable instruction. The validator knows the schema — the domain agent does not.

In [59]:
def node_fe_validator(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 8: FE VALIDATOR")
    print("="*60)

    prompt = f"""You are a technical translator for a PySpark feature engineering pipeline.
A feature engineering agent has described transformations in plain English.
Your job is to translate each description into a precise executable instruction.

The feature engineering agent's decisions:
{json.dumps(state['fe_decisions_raw'], indent=2)}

Translate each transformation into one of the following executable instruction types.
Choose the best-matching type for each description.

Supported instruction types and their required fields:

- log1p:             apply log(1+x) to handle right-skewed data
  {{"type":"log1p","input":"col","output":"out_col"}}

- sqrt:              apply sqrt(x) as a milder skew reduction
  {{"type":"sqrt","input":"col","output":"out_col"}}

- ratio:             (numerator + smoothing) / (denominator + smoothing)
  {{"type":"ratio","numerator":"col_a","denominator":"col_b","output":"out_col","smoothing":1}}

- difference:        col_a - col_b
  {{"type":"difference","col_a":"x","col_b":"y","output":"out_col"}}

- binary_flag:       1 if condition is true, else 0
  {{"type":"binary_flag","input":"col","condition":"gt|gte|lt|lte|eq|neq","threshold":0,"output":"out_col"}}

- text_length:       length of text in chars, words, or sentences
  {{"type":"text_length","input":"col","unit":"chars|words|sentences","output":"out_col"}}

- text_count:        count occurrences of a regex pattern in text
  {{"type":"text_count","input":"col","pattern":"!","output":"out_col"}}

- missingness_flag:  1 if null, else 0
  {{"type":"missingness_flag","input":"col","output":"out_col"}}

- tfidf:             TF-IDF sparse vector from text column
  {{"type":"tfidf","input":"col","output":"out_col","num_features":5000}}

- datetime_extract:  extract year/month/dayofweek/hour from unix timestamp
  {{"type":"datetime_extract","input":"ts_col","extract":"year|month|dayofweek|hour","output":"out_col"}}

- time_diff:         difference between two unix timestamp columns in seconds
  {{"type":"time_diff","col_a":"ts1","col_b":"ts2","output":"out_col"}}

- aggregate_by_group: aggregate a column grouped by another (leakage-safe: fit on train only)
  {{"type":"aggregate_by_group","group_col":"app_name","agg_col":"recommended","agg_func":"mean|count|sum","output":"out_col","leakage_safe":true}}

- passthrough:       include a raw column as-is (cast to double)
  {{"type":"passthrough","input":"col","output":"out_col"}}

- drop_column:       remove a column entirely (e.g. raw text after feature extraction)
  {{"type":"drop_column","input":"col"}}

For any transformation you cannot map to these types, include it in 'unresolvable'.

Return ONLY valid JSON:
{{
  "validated_transformations": [
    {{"type": "log1p", "input": "votes_helpful", "output": "log1p_votes_helpful"}}
  ],
  "unresolvable": [
    {{"original_description": "...", "reason": "why it cannot be mapped"}}
  ],
  "translation_notes": "How you mapped each raw decision to the schema."
}}"""

    result = call_agent(prompt, "FE VALIDATOR — TRANSLATED SCHEMA")
    validated = result.get("validated_transformations", [])
    unresolvable = result.get("unresolvable", [])

    print(f"\n  Validated: {len(validated)} | Unresolvable: {len(unresolvable)}")
    for i, t in enumerate(validated, 1):
        print(f"  {i:2d}. [{t.get('type'):<20}] -> {t.get('output', t.get('input','?'))}")
    if unresolvable:
        print(f"  Unresolvable:")
        for u in unresolvable:
            print(f"    - {u.get('original_description','')[:80]}")

    state["fe_decisions_validated"] = result
    return state

### Node 9: Apply Feature Engineering

PySpark dynamic dispatcher executes each validated instruction. Train/valid split happens first (before any leakage-sensitive transforms).

In [60]:
def node_apply_fe(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 9: APPLY FEATURE ENGINEERING")
    print("="*60)

    df = state["df"]
    instructions = state["fe_decisions_validated"].get("validated_transformations", [])
    skipped = []

    # Train/valid split FIRST
    train_df, valid_df = df.randomSplit([0.8, 0.2], seed=42)
    train_df.cache()
    valid_df.cache()
    print(f"Train: {train_df.count():,} | Valid: {valid_df.count():,}")

    def both(fn):
        nonlocal train_df, valid_df
        train_df, valid_df = fn(train_df), fn(valid_df)

    agg_cache = {}  # for leakage-safe aggregates

    for instr in instructions:
        t   = instr.get("type")
        out = instr.get("output")

        if t == "drop_column":
            continue

        try:
            if t == "log1p":
                c = instr["input"]
                both(lambda d,c=c,o=out: d.withColumn(o, F.log1p(F.coalesce(F.col(c), F.lit(0)))))
                print(f"  [log1p] {c} -> {out}")

            elif t == "sqrt":
                c = instr["input"]
                both(lambda d,c=c,o=out: d.withColumn(o, F.sqrt(F.coalesce(F.col(c), F.lit(0)))))
                print(f"  [sqrt] {c} -> {out}")

            elif t == "ratio":
                n, de, sm = instr["numerator"], instr["denominator"], instr.get("smoothing", 1)
                both(lambda d,n=n,de=de,sm=sm,o=out:
                    d.withColumn(o, (F.coalesce(F.col(n),F.lit(0))+sm) / (F.coalesce(F.col(de),F.lit(0))+sm)))
                print(f"  [ratio] ({n}+{sm})/({de}+{sm}) -> {out}")

            elif t == "difference":
                a, b = instr["col_a"], instr["col_b"]
                both(lambda d,a=a,b=b,o=out:
                    d.withColumn(o, F.coalesce(F.col(a),F.lit(0)) - F.coalesce(F.col(b),F.lit(0))))
                print(f"  [diff] {a} - {b} -> {out}")

            elif t == "binary_flag":
                c, cond, thr = instr["input"], instr.get("condition","gt"), instr.get("threshold",0)
                ops = {"gt":">","gte":">=","lt":"<","lte":"<=","eq":"==","neq":"!="}
                op = ops.get(cond, ">")
                dtype = dict(df.dtypes).get(c, dict(train_df.dtypes).get(c, "string"))
                if dtype == "string":
                    # agent likely meant length of the string, not the string itself
                    both(lambda d,c=c,op=op,thr=thr,o=out:
                        d.withColumn(o, F.when(F.expr(f"length(`{c}`) {op} {thr}"),1).otherwise(0)))
                    print(f"  [flag] length({c}) {cond} {thr} -> {out} (string col: applied length())")
                else:
                    both(lambda d,c=c,op=op,thr=thr,o=out:
                        d.withColumn(o, F.when(F.expr(f"`{c}` {op} {thr}"),1).otherwise(0)))
                    print(f"  [flag] {c} {cond} {thr} -> {out}")

            elif t == "text_length":
                c, unit = instr["input"], instr.get("unit","chars")
                if unit == "chars":
                    both(lambda d,c=c,o=out: d.withColumn(o, F.length(F.col(c))))
                elif unit == "words":
                    both(lambda d,c=c,o=out: d.withColumn(o, F.size(F.split(F.trim(F.col(c)), r"\s+"))))
                elif unit == "sentences":
                    both(lambda d,c=c,o=out: d.withColumn(o, F.size(F.split(F.col(c), r"[.!?]+"))))
                print(f"  [text_length:{unit}] {c} -> {out}")

            elif t == "text_count":
                import re
                c, pat = instr["input"], instr.get("pattern","!")
                safe_pat = re.escape(pat)
                both(lambda d,c=c,pat=safe_pat,o=out:
                    d.withColumn(o, F.length(F.col(c)) - F.length(F.regexp_replace(F.col(c), pat, ""))))
                print(f"  [text_count] pattern='{pat}' in {c} -> {out}")

            elif t == "missingness_flag":
                c = instr["input"]
                both(lambda d,c=c,o=out: d.withColumn(o, F.when(F.col(c).isNull(),1).otherwise(0)))
                print(f"  [missing_flag] {c} -> {out}")

            elif t == "passthrough":
                c = instr["input"]
                both(lambda d,c=c,o=out: d.withColumn(o, F.col(c).cast(DoubleType())))
                print(f"  [passthrough] {c} -> {out}")

            elif t == "datetime_extract":
                c, ext = instr["input"], instr.get("extract","year")
                fn_map = {"year":F.year,"month":F.month,"dayofweek":F.dayofweek,"hour":F.hour}
                fn = fn_map.get(ext, F.year)
                both(lambda d,c=c,fn=fn,o=out:
                    d.withColumn(o, fn(F.from_unixtime(F.col(c)).cast("timestamp"))))
                print(f"  [datetime:{ext}] {c} -> {out}")

            elif t == "time_diff":
                a, b = instr["col_a"], instr["col_b"]
                both(lambda d,a=a,b=b,o=out:
                    d.withColumn(o, F.coalesce(
                        F.when(F.col(a).isNotNull() & F.col(b).isNotNull(), F.col(a)-F.col(b)),
                        F.lit(0))))
                print(f"  [time_diff] {a}-{b} -> {out}")

            elif t == "tfidf":
                c, n = instr["input"], instr.get("num_features",5000)
                tok = Tokenizer(inputCol=c, outputCol="_tok_raw")
                rem = StopWordsRemover(inputCol="_tok_raw", outputCol="_tok_clean")
                htf = HashingTF(inputCol="_tok_clean", outputCol="_tf", numFeatures=n)
                idf = IDF(inputCol="_tf", outputCol=out)
                for step in [tok, rem, htf]:
                    train_df = step.transform(train_df)
                    valid_df = step.transform(valid_df)
                idf_m = idf.fit(train_df)
                train_df = idf_m.transform(train_df)
                valid_df = idf_m.transform(valid_df)
                print(f"  [tfidf:{n}] {c} -> {out} (leakage-safe)")

            elif t == "aggregate_by_group":
                grp, agg_col = instr["group_col"], instr["agg_col"]
                agg_func = instr.get("agg_func","mean")
                leakage  = instr.get("leakage_safe", True)
                key = f"{grp}__{agg_col}__{agg_func}"
                if leakage and key not in agg_cache:
                    fn_map = {"mean":F.mean,"count":F.count,"sum":F.sum,"max":F.max,"min":F.min}
                    agg_fn = fn_map.get(agg_func, F.mean)
                    stats = train_df.groupBy(grp).agg(agg_fn(agg_col).alias(out))
                    default = float(stats.select(F.mean(out)).collect()[0][0] or 0)
                    agg_cache[key] = (stats, default)
                if key in agg_cache:
                    stats, default = agg_cache[key]
                    if out not in train_df.columns:
                        train_df = train_df.join(stats, on=grp, how="left").fillna({out: default})
                        valid_df = valid_df.join(stats, on=grp, how="left").fillna({out: default})
                        print(f"  [agg:{agg_func}] {agg_col} by {grp} -> {out} (leakage_safe={leakage})")
                    else:
                        print(f"  [SKIP agg] {out} already exists, skipping to avoid collision")

            else:
                skipped.append(instr)
                print(f"  [SKIP] Unknown type: {t}")

        except Exception as e:
            skipped.append({**instr, "error": str(e)})
            print(f"  [ERROR] {t} -> {out}: {e}")

    # Execute drop_column instructions last
    for instr in instructions:
        if instr.get("type") == "drop_column":
            c = instr["input"]
            if c in train_df.columns:
                both(lambda d, c=c: d.drop(c))
                print(f"  [drop_column] {c}")

    # Build dense feature list
    dense_feature_cols = [
        instr.get("output", instr.get("input"))
        for instr in instructions
        if instr.get("type") != "tfidf"
        and instr.get("type") != "drop_column"
        and instr.get("output", instr.get("input")) in train_df.columns
    ]
    dense_feature_cols = list(dict.fromkeys(dense_feature_cols))  # deduplicate, preserve order

    print(f"\nDense features produced: {len(dense_feature_cols)}")
    print(f"Skipped: {len(skipped)}")

    state["train_df"]           = train_df
    state["valid_df"]           = valid_df
    state["dense_feature_cols"] = dense_feature_cols
    state["skipped_instructions"] = skipped
    return state

### Node 10: Compute Post-FE Summary

Brief summary fed to the Model Config Agent.

In [61]:
def node_fe_summary(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 10: POST-FE SUMMARY")
    print("="*60)

    train_df = state["train_df"]
    valid_df = state["valid_df"]
    dense    = state["dense_feature_cols"]

    tr_count = train_df.count()
    va_count = valid_df.count()
    tr_dist  = {int(r["recommended"]): int(r["count"]) for r in train_df.groupBy("recommended").count().collect()}
    va_dist  = {int(r["recommended"]): int(r["count"]) for r in valid_df.groupBy("recommended").count().collect()}

    has_tfidf = "tfidf_features" in train_df.columns
    imbalance = round(max(tr_dist.values()) / max(min(tr_dist.values()), 1), 4)

    fe_summary = {
        "train_rows": int(tr_count),
        "valid_rows": int(va_count),
        "dense_feature_count": len(dense),
        "dense_feature_cols": dense,
        "has_tfidf_features": has_tfidf,
        "tfidf_dimensions": 5000 if has_tfidf else 0,
        "class_distribution_train": tr_dist,
        "class_distribution_valid": va_dist,
        "imbalance_ratio": imbalance,
        "skipped_instructions_count": len(state.get("skipped_instructions", [])),
        "task": "Binary classification: predict recommended (1=positive, 0=negative)",
        "domain": "Steam game reviews — text + metadata features"
    }

    print(f"Train: {tr_count:,} | Valid: {va_count:,}")
    print(f"Dense features: {len(dense)} | TF-IDF: {has_tfidf} | Imbalance: {imbalance}")

    state["fe_summary"] = fe_summary
    return state

### Node 11: Model Config Agent

Receives post-FE summary. Reasons freely about which models to use and what hyperparameter configurations to try. No schema shown.

In [62]:
def node_model_config_agent(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 11: MODEL CONFIG AGENT")
    print("="*60)

    prompt = f"""You are a machine learning expert configuring models for a classification pipeline.

EXPERIMENT GOAL: The primary objective of this experiment is not just to achieve high prediction
accuracy, but to UNDERSTAND what drives a Steam review to be positive or negative.
Feature importance analysis is central — the selected models must produce interpretable
feature importance so we can identify which factors most strongly influence review classification.
Models that cannot produce feature importance (e.g. black-box ensembles without coefficients
or importance scores) are unsuitable for this experiment.

The task is binary classification: predict recommended (1=positive, 0=negative) from Steam review
metadata and text features.

You have been given a summary of the feature-engineered dataset.
Based purely on what you observe, decide:

1. Which two ML algorithms to train.
   Think broadly about classical ML — linear models, tree-based models, ensemble methods, etc.
   Choose models that are:
   (a) well-suited to this data and task, AND
   (b) capable of producing interpretable feature importance (e.g. coefficients or Gini importance).
   Do NOT constrain yourself to any fixed list — reason about what fits best.

2. For each algorithm, decide which hyperparameters matter most and provide 4-6 specific
   configurations to try. Think about the dataset characteristics (size, feature count,
   class balance, presence of both dense and sparse TF-IDF features) when choosing ranges.

3. What evaluation metric to use as the primary selection criterion.

4. Whether class imbalance needs special handling.

Feature-engineered dataset summary:
{json.dumps(state["fe_summary"], indent=2)}

Return your decisions in whatever JSON structure makes sense to you.
Include: "reasoning", "model_1", "model_2", "primary_metric", "primary_metric_justification", "handle_imbalance".

{{
  "reasoning": "Your full reasoning...",
  "model_1": {{
    "name": "algorithm name in your own words",
    "why_chosen": "justification referencing data characteristics AND interpretability for feature importance",
    "hyperparameter_configs": [
      {{... whatever fields you think are relevant for this algorithm ...}}
    ]
  }},
  "model_2": {{
    "name": "algorithm name in your own words",
    "why_chosen": "justification",
    "hyperparameter_configs": [
      {{... configs ...}}
    ]
  }},
  "primary_metric": "metric name",
  "primary_metric_justification": "why this metric suits the task",
  "handle_imbalance": false
}}"""

    result = call_agent(prompt, "MODEL CONFIG AGENT — RAW DECISIONS")
    print(f"\n  Model 1: {result.get('model_1',{}).get('name')}")
    print(f"  Model 2: {result.get('model_2',{}).get('name')}")
    print(f"  Primary metric: {result.get('primary_metric')}")
    print(f"  M1 configs: {len(result.get('model_1',{}).get('hyperparameter_configs',[]))}")
    print(f"  M2 configs: {len(result.get('model_2',{}).get('hyperparameter_configs',[]))}")

    state["model_config_raw"] = result
    return state

### Node 12: Model Config Validator

Translates the Model Config Agent's free-form decisions into PySpark-executable model configurations.

In [63]:
def node_model_config_validator(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 12: MODEL CONFIG VALIDATOR")
    print("="*60)

    prompt = f"""You are a technical translator for a PySpark ML pipeline.
A model configuration agent has freely chosen ML algorithms and hyperparameters.
Your job is to:
1. Determine whether each algorithm is available in PySpark MLlib
2. Map its hyperparameters to the correct PySpark field names and valid value types
3. Flag anything that cannot be supported with a clear explanation

SUPPORTED PySpark MLlib classifiers and their exact hyperparameter field names:

  LogisticRegression:
    maxIter (int), regParam (float), elasticNetParam (float 0-1, 0=L2 1=L1)
    -> feature importance: model.coefficients (one per feature, sign = direction)

  RandomForestClassifier:
    numTrees (int), maxDepth (int), featureSubsetStrategy (str: "sqrt","log2","auto","all"),
    minInstancesPerNode (int)
    -> feature importance: model.featureImportances (Gini importance)

  GBTClassifier:
    maxIter (int), maxDepth (int), stepSize (float)
    -> feature importance: model.featureImportances (Gini importance)

  DecisionTreeClassifier:
    maxDepth (int), minInstancesPerNode (int)
    -> feature importance: model.featureImportances (Gini importance)

  LinearSVC:
    maxIter (int), regParam (float)
    -> feature importance: model.coefficients (magnitude = importance, no probability output)

  NaiveBayes:
    smoothing (float), modelType (str: "multinomial" or "bernoulli")
    -> feature importance: model.theta (log probabilities per class per feature)
    NOTE: requires all feature values to be non-negative

UNSUPPORTED (not in PySpark MLlib): KNN, SVM with RBF kernel, XGBoost (unless custom),
  neural networks (use ML lib for those separately), sklearn models.

If the agent chose an unsupported algorithm, map it to the closest supported one and explain why.
If a hyperparameter field name doesn't match, map it to the correct PySpark field name.
If a hyperparameter value is out of valid range, clip it to the nearest valid value.

The model config agent's decisions:
{json.dumps(state["model_config_raw"], indent=2)}

Return ONLY valid JSON:
{{
  "model_1": {{
    "pyspark_class": "exact PySpark class name",
    "original_name": "what the agent called it",
    "configs": [
      {{... config with correct PySpark field names ...}}
    ],
    "mapping_notes": "how you mapped the agent's choice to PySpark"
  }},
  "model_2": {{
    "pyspark_class": "exact PySpark class name",
    "original_name": "what the agent called it",
    "configs": [
      {{... config ...}}
    ],
    "mapping_notes": "how you mapped the agent's choice to PySpark"
  }},
  "primary_metric": "areaUnderROC",
  "primary_metric_evaluator": "binary",
  "secondary_metrics": ["f1", "accuracy"],
  "unresolvable": [
    {{"original_decision": "...", "reason": "why it cannot be supported"}}
  ],
  "translation_notes": "Overall translation notes."
}}"""

    result = call_agent(prompt, "MODEL CONFIG VALIDATOR — TRANSLATED SCHEMA")
    print(f"\n  M1: {result.get('model_1',{}).get('original_name')} -> {result.get('model_1',{}).get('pyspark_class')} ({len(result.get('model_1',{}).get('configs',[]))} configs)")
    print(f"  M2: {result.get('model_2',{}).get('original_name')} -> {result.get('model_2',{}).get('pyspark_class')} ({len(result.get('model_2',{}).get('configs',[]))} configs)")
    print(f"  Primary metric: {result.get('primary_metric')} ({result.get('primary_metric_evaluator')})")
    if result.get("unresolvable"):
        print(f"  Unresolvable: {[u.get('original_decision','')[:60] for u in result['unresolvable']]}")

    state["model_config_validated"] = result
    return state

### Node 13: Train Models

PySpark trains both models using the validated configurations. Picks best per model by primary metric. Extracts feature importance.

In [64]:
def node_train_models(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 13: TRAIN MODELS")
    print("="*60)

    train_df  = state["train_df"]
    valid_df  = state["valid_df"]
    dense     = state["dense_feature_cols"]
    cfg       = state["model_config_validated"]
    LABEL_COL = "recommended"
    TEXT_COL  = "tfidf_features"
    SEED      = 42
    pri_met   = cfg.get("primary_metric", "areaUnderROC")
    pri_eval  = cfg.get("primary_metric_evaluator", "binary")

    # Cast + impute + assemble
    for c in dense:
        train_df = train_df.withColumn(c, F.col(c).cast(DoubleType()))
        valid_df = valid_df.withColumn(c, F.col(c).cast(DoubleType()))
    train_df = train_df.withColumn(LABEL_COL, F.col(LABEL_COL).cast(DoubleType()))
    valid_df = valid_df.withColumn(LABEL_COL, F.col(LABEL_COL).cast(DoubleType()))

    dense = [c for c in dense if c in train_df.columns]
    imp_cols = [f"{c}_imp" for c in dense]
    imp_model = Imputer(inputCols=dense, outputCols=imp_cols, strategy="mean").fit(train_df)
    train_df  = imp_model.transform(train_df)
    valid_df  = imp_model.transform(valid_df)

    asm_inputs = imp_cols + ([TEXT_COL] if TEXT_COL in train_df.columns else [])
    assembler  = VectorAssembler(inputCols=asm_inputs, outputCol="features", handleInvalid="skip")
    tr_asm = assembler.transform(train_df).select(LABEL_COL, "features").cache()
    va_asm = assembler.transform(valid_df).select(LABEL_COL, "features").cache()

    # Evaluators
    bin_evals = {
        "areaUnderROC": BinaryClassificationEvaluator(labelCol=LABEL_COL, metricName="areaUnderROC"),
        "areaUnderPR":  BinaryClassificationEvaluator(labelCol=LABEL_COL, metricName="areaUnderPR"),
    }
    mul_evals = {
        "f1":                MulticlassClassificationEvaluator(labelCol=LABEL_COL, metricName="f1"),
        "accuracy":          MulticlassClassificationEvaluator(labelCol=LABEL_COL, metricName="accuracy"),
        "weightedPrecision": MulticlassClassificationEvaluator(labelCol=LABEL_COL, metricName="weightedPrecision"),
        "weightedRecall":    MulticlassClassificationEvaluator(labelCol=LABEL_COL, metricName="weightedRecall"),
    }
    all_evals = {**bin_evals, **mul_evals}

    MODEL_MAP = {"LogisticRegression": LogisticRegression, "RandomForestClassifier": RandomForestClassifier}
    try:
        from pyspark.ml.classification import (
            GBTClassifier, LinearSVC, DecisionTreeClassifier, NaiveBayes,
            MultilayerPerceptronClassifier, FMClassifier
        )
        MODEL_MAP.update({
            "GBTClassifier":                   GBTClassifier,
            "LinearSVC":                       LinearSVC,
            "DecisionTreeClassifier":          DecisionTreeClassifier,
            "NaiveBayes":                      NaiveBayes,
            "MultilayerPerceptronClassifier":  MultilayerPerceptronClassifier,
            "FMClassifier":                    FMClassifier,
        })
    except Exception as e:
        print(f"  Note: some classifiers not available in this Spark version: {e}")

    def run_model(model_spec):
        cls_name = model_spec.get("pyspark_class")
        configs  = model_spec.get("configs", [])
        cls      = MODEL_MAP.get(cls_name)
        if not cls:
            print(f"  [SKIP] {cls_name} not available")
            return [], None, None
        results = []
        print(f"\n  {cls_name} ({len(configs)} configs)")
        for cfg_item in configs:
            try:
                m = cls(featuresCol="features", labelCol=LABEL_COL, **cfg_item)
                if hasattr(m, "setSeed"): m = m.setSeed(SEED)
                fitted = m.fit(tr_asm)
                preds  = fitted.transform(va_asm)
                scores = {}
                for met, ev in all_evals.items():
                    try: scores[met] = round(float(ev.evaluate(preds)), 4)
                    except Exception: scores[met] = None
                row = {**cfg_item, **scores, "_model": fitted}
                results.append(row)
                print(f"    {cfg_item} -> {pri_met}={scores.get(pri_met)}")
            except Exception as e:
                print(f"    [ERROR] {cfg_item}: {e}")
        if not results: return [], None, None
        best = max(results, key=lambda x: x.get(pri_met) or 0)
        return [{k:v for k,v in r.items() if k!="_model"} for r in results], {k:v for k,v in best.items() if k!="_model"}, best["_model"]

    m1_res, m1_best, m1_obj = run_model(cfg.get("model_1", {}))
    m2_res, m2_best, m2_obj = run_model(cfg.get("model_2", {}))

    m1_name = cfg.get("model_1", {}).get("pyspark_class", "Model1")
    m2_name = cfg.get("model_2", {}).get("pyspark_class", "Model2")
    print(f"\nBest {m1_name}: {pri_met}={m1_best.get(pri_met) if m1_best else 'N/A'}")
    print(f"Best {m2_name}: {pri_met}={m2_best.get(pri_met) if m2_best else 'N/A'}")

    # Feature importance
    def extract_fi(model_obj, dense_cols):
        if model_obj is None: return {}
        nd = len(dense_cols)
        try:
            if hasattr(model_obj, "coefficients"):
                coeffs = model_obj.coefficients.toArray()
                dense_part, text_part = coeffs[:nd], coeffs[nd:]
                top10 = sorted([{"feature":dense_cols[i],"coefficient":round(float(dense_part[i]),6),"abs":round(abs(float(dense_part[i])),6)} for i in range(nd)], key=lambda x:x["abs"], reverse=True)[:10]
                dt, tt = float(np.sum(np.abs(dense_part))), float(np.sum(np.abs(text_part)))
                tot = dt+tt
                return {"top10_dense":top10, "dense_share":round(dt/tot,4) if tot>0 else 0, "tfidf_share":round(tt/tot,4) if tot>0 else 0}
            elif hasattr(model_obj, "featureImportances"):
                imps = model_obj.featureImportances.toArray()
                dense_part, text_part = imps[:nd], imps[nd:]
                top10 = sorted([{"feature":dense_cols[i],"importance":round(float(dense_part[i]),6)} for i in range(nd)], key=lambda x:x["importance"], reverse=True)[:10]
                dt, tt = float(np.sum(dense_part)), float(np.sum(text_part))
                tot = dt+tt
                return {"top10_dense":top10, "dense_share":round(dt/tot,4) if tot>0 else 0, "tfidf_share":round(tt/tot,4) if tot>0 else 0}
            elif hasattr(model_obj, "theta"):
                # NaiveBayes — use difference in log-probabilities between classes as proxy importance
                theta = model_obj.theta.toArray()
                if theta.shape[0] == 2:
                    diff = np.abs(theta[1] - theta[0])
                    n = min(len(dense_cols), len(diff))
                    top10 = sorted(
                        [{"feature": dense_cols[i], "importance": round(float(diff[i]), 6)} for i in range(n)],
                        key=lambda x: x["importance"], reverse=True
                    )[:10]
                    return {"top10_dense": top10, "dense_share": None, "tfidf_share": None,
                            "note": "NaiveBayes importance approximated from class log-probability differences"}
        except Exception as e:
            return {"error": str(e)}
        return {}

    fi = {
        "model_1_name": m1_name, "model_1": extract_fi(m1_obj, dense),
        "model_2_name": m2_name, "model_2": extract_fi(m2_obj, dense),
    }
    m1_top = set(x["feature"] for x in fi["model_1"].get("top10_dense", []))
    m2_top = set(x["feature"] for x in fi["model_2"].get("top10_dense", []))
    fi["overlapping_top10"] = list(m1_top & m2_top)
    print(f"Overlapping top-10: {fi['overlapping_top10']}")

    state["model_1_results"] = m1_res
    state["model_2_results"] = m2_res
    state["model_1_best"]    = m1_best
    state["model_2_best"]    = m2_best
    state["feature_importance"] = fi
    return state

### Node 14: Evaluation Agent

Receives all model results and feature importance data. Produces domain-grounded analysis contextualised to Steam reviews. No validator needed — output is narrative, not executed.

In [65]:
def node_evaluation_agent(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 14: EVALUATION AGENT")
    print("="*60)

    cfg     = state["model_config_validated"]
    m1_name = cfg.get("model_1", {}).get("pyspark_class", "Model1")
    m2_name = cfg.get("model_2", {}).get("pyspark_class", "Model2")
    pri_met = cfg.get("primary_metric", "areaUnderROC")

    eval_input = {
        "experiment_goal": (
            "The PRIMARY goal is not just accuracy — it is to understand WHAT DRIVES a Steam review "
            "to be positive or negative. Feature importance analysis is the centrepiece of this experiment. "
            "The evaluation should focus on what the feature importance results reveal about review behaviour, "
            "grounded in Steam domain knowledge."
        ),
        "task": "Predict whether a Steam game review is positive (recommended=1) or negative (recommended=0)",
        "domain_context": (
            "Steam is a PC gaming platform. Reviews are written by players after purchasing/playing games. "
            "A positive review means the player recommends the game. "
            "Metadata includes: playtime (how long they played), votes_helpful (community upvotes), "
            "votes_funny (joke/meme votes), weighted_vote_score (Steam's internal helpfulness score), "
            "comment_count (discussion activity), steam_purchase (bought on Steam vs key), "
            "received_for_free (review copy), written_during_early_access (game not fully released)."
        ),
        "primary_metric": pri_met,
        "model_1_name": m1_name,
        "model_1_all_configs": state["model_1_results"],
        "model_1_best": state["model_1_best"],
        "model_1_feature_importance": state["feature_importance"].get("model_1", {}),
        "model_2_name": m2_name,
        "model_2_all_configs": state["model_2_results"],
        "model_2_best": state["model_2_best"],
        "model_2_feature_importance": state["feature_importance"].get("model_2", {}),
        "overlapping_top10_features": state["feature_importance"].get("overlapping_top10", []),
        "class_distribution": state["fe_summary"]["class_distribution_train"],
        "dense_features_used": state["dense_feature_cols"],
        "has_tfidf": state["fe_summary"].get("has_tfidf_features", False),
    }

    prompt = f"""You are an evaluation expert analysing the results of a Steam review classification pipeline.

EXPERIMENT GOAL: The primary objective was to understand WHAT DRIVES a Steam review to be positive
or negative, using feature importance as the key analytical lens. Your evaluation must centre on this.

You have complete model training results and feature importance data.
Produce a thorough evaluation that:

1. Compares both models on performance metrics — which is better and why structurally
2. CENTRES on feature importance analysis as the main finding:
   - Which features are most important across both models?
   - What does each important feature tell us about review behaviour in Steam domain terms?
     e.g. What does high playtime at review time tell us? What do helpful votes signal?
     What does app_recommend_rate reveal about collective sentiment influence?
   - Where do the two models agree or disagree on feature importance, and what does that mean?
   - What does the dense features vs TF-IDF share tell us — is this primarily a text problem
     or a metadata problem?
3. Directly and specifically answers: what are the strongest factors driving Steam review classification?
4. Gives practical implications for game developers and the Steam platform based on these findings
5. Honestly discusses limitations of this analysis

Full results and context:
{json.dumps(eval_input, indent=2)}

Return ONLY valid JSON:
{{
  "selected_model": "...",
  "selected_model_metric": 0.00,
  "runner_up_model": "...",
  "runner_up_metric": 0.00,
  "metric_name": "...",
  "metric_gap": 0.00,
  "model_comparison": "2-3 sentences",
  "feature_importance_analysis": "3-4 sentences — the main finding, centred on what each top feature means in Steam domain terms",
  "feature_importance_deep_dive": [
    {{"feature": "feature_name", "model_1_rank": 1, "model_2_rank": 2, "interpretation": "what this feature reveals about review behaviour"}}
  ],
  "dense_vs_tfidf_interpretation": "1-2 sentences — is this primarily a text problem or a metadata problem?",
  "domain_insights": ["insight 1 grounded in Steam domain", "insight 2", "insight 3"],
  "answer_to_problem_statement": "Direct specific answer: what are the strongest factors driving Steam review classification?",
  "limitations": ["limitation 1", "limitation 2", "limitation 3"],
  "practical_implications": "What should game developers or Steam do with these findings?",
  "narrative": "3-4 paragraph cohesive summary for a project report.",
  "reasoning": "Your evaluation reasoning process."
}}"""

    result = call_agent(prompt, "EVALUATION AGENT")

    print("\n--- EVALUATION SUMMARY ---")
    print(f"  Selected: {result.get('selected_model')} ({pri_met}={result.get('selected_model_metric')})")
    print(f"  Runner-up: {result.get('runner_up_model')} ({pri_met}={result.get('runner_up_metric')})")
    print(f"  Gap: {result.get('metric_gap')}")
    print("\n  Domain insights:")
    for ins in result.get("domain_insights", []):
        print(f"    - {ins}")
    print("\n  Answer to problem statement:")
    print(f"    {result.get('answer_to_problem_statement')}")
    print("\n  Narrative:")
    print(result.get("narrative", ""))

    state["evaluation_decisions"] = result
    return state

### Node 15: Export All Outputs

In [66]:
def node_export(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("NODE 15: EXPORT")
    print("="*60)

    with open("agent_decisions.json", "w") as f:
        json.dump({
            "cleaning": {
                "raw":       state["cleaning_decisions_raw"],
                "validated": state["cleaning_decisions_validated"],
            },
            "feature_engineering": {
                "raw":       state["fe_decisions_raw"],
                "validated": state["fe_decisions_validated"],
                "dense_feature_cols": state["dense_feature_cols"],
                "skipped_instructions": state.get("skipped_instructions", []),
            },
            "model_config": {
                "raw":       state["model_config_raw"],
                "validated": state["model_config_validated"],
            },
        }, f, indent=2)
    print("  Saved: agent_decisions.json")

    cfg     = state["model_config_validated"]
    m1_name = cfg.get("model_1", {}).get("pyspark_class", "Model1")
    m2_name = cfg.get("model_2", {}).get("pyspark_class", "Model2")

    with open("evaluation_report.json", "w") as f:
        json.dump({
            f"{m1_name}_all_results": state["model_1_results"],
            f"{m2_name}_all_results": state["model_2_results"],
            f"{m1_name}_best":        state["model_1_best"],
            f"{m2_name}_best":        state["model_2_best"],
            "feature_importance":     state["feature_importance"],
            "evaluation":             state["evaluation_decisions"],
        }, f, indent=2)
    print("  Saved: evaluation_report.json")

    state["output_json_path"]       = "agent_decisions.json"
    state["evaluation_report_path"] = "evaluation_report.json"
    return state

## 6. Build & Compile LangGraph Pipeline

```
load_data -> raw_summary
    -> cleaning_agent    (reasons freely)
    -> cleaning_validator (translates to schema)
    -> apply_cleaning
    -> cleaned_summary
    -> fe_agent          (reasons freely)
    -> fe_validator      (translates to schema)
    -> apply_fe
    -> fe_summary
    -> model_config_agent    (reasons freely)
    -> model_config_validator (translates to schema)
    -> train_models
    -> evaluation_agent  (reasons freely, no validator needed)
    -> export -> END
```

In [67]:
builder = StateGraph(AgentState)

builder.add_node("load_data",               node_load_data)
builder.add_node("raw_summary",             node_raw_summary)
builder.add_node("cleaning_agent",          node_cleaning_agent)
builder.add_node("cleaning_validator",      node_cleaning_validator)
builder.add_node("apply_cleaning",          node_apply_cleaning)
builder.add_node("cleaned_summary",         node_cleaned_summary)
builder.add_node("fe_agent",               node_fe_agent)
builder.add_node("fe_validator",           node_fe_validator)
builder.add_node("apply_fe",              node_apply_fe)
builder.add_node("fe_summary",            node_fe_summary)
builder.add_node("model_config_agent",     node_model_config_agent)
builder.add_node("model_config_validator", node_model_config_validator)
builder.add_node("train_models",           node_train_models)
builder.add_node("evaluation_agent",       node_evaluation_agent)
builder.add_node("export",                node_export)

builder.set_entry_point("load_data")
builder.add_edge("load_data",               "raw_summary")
builder.add_edge("raw_summary",             "cleaning_agent")
builder.add_edge("cleaning_agent",          "cleaning_validator")
builder.add_edge("cleaning_validator",      "apply_cleaning")
builder.add_edge("apply_cleaning",          "cleaned_summary")
builder.add_edge("cleaned_summary",         "fe_agent")
builder.add_edge("fe_agent",               "fe_validator")
builder.add_edge("fe_validator",           "apply_fe")
builder.add_edge("apply_fe",              "fe_summary")
builder.add_edge("fe_summary",            "model_config_agent")
builder.add_edge("model_config_agent",     "model_config_validator")
builder.add_edge("model_config_validator", "train_models")
builder.add_edge("train_models",           "evaluation_agent")
builder.add_edge("evaluation_agent",       "export")
builder.add_edge("export",                END)

pipeline = builder.compile()

print("LangGraph pipeline compiled.")
print()
nodes = [
    (" 1", "load_data",               "non-agent prep"),
    (" 2", "raw_summary",             "feeds Cleaning Agent"),
    (" 3", "cleaning_agent",          "<- GPT: reasons freely from raw stats"),
    (" 4", "cleaning_validator",      "<- GPT: translates to executable schema"),
    (" 5", "apply_cleaning",          "<- PySpark executes validated cleaning"),
    (" 6", "cleaned_summary",         "deep per-feature stats, feeds FE Agent"),
    (" 7", "fe_agent",               "<- GPT: reasons freely, no schema shown"),
    (" 8", "fe_validator",           "<- GPT: translates to executable schema"),
    (" 9", "apply_fe",              "<- PySpark dynamic dispatcher"),
    ("10", "fe_summary",            "feeds Model Config Agent"),
    ("11", "model_config_agent",     "<- GPT: reasons freely"),
    ("12", "model_config_validator", "<- GPT: translates to PySpark schema"),
    ("13", "train_models",           "<- PySpark trains + evaluates + feature importance"),
    ("14", "evaluation_agent",       "<- GPT: domain-grounded analysis"),
    ("15", "export",                "saves agent_decisions.json + evaluation_report.json"),
]
for num, name, note in nodes:
    print(f"  {num}. {name:<24} {note}")

LangGraph pipeline compiled.

   1. load_data                non-agent prep
   2. raw_summary              feeds Cleaning Agent
   3. cleaning_agent           <- GPT: reasons freely from raw stats
   4. cleaning_validator       <- GPT: translates to executable schema
   5. apply_cleaning           <- PySpark executes validated cleaning
   6. cleaned_summary          deep per-feature stats, feeds FE Agent
   7. fe_agent                 <- GPT: reasons freely, no schema shown
   8. fe_validator             <- GPT: translates to executable schema
   9. apply_fe                 <- PySpark dynamic dispatcher
  10. fe_summary               feeds Model Config Agent
  11. model_config_agent       <- GPT: reasons freely
  12. model_config_validator   <- GPT: translates to PySpark schema
  13. train_models             <- PySpark trains + evaluates + feature importance
  14. evaluation_agent         <- GPT: domain-grounded analysis
  15. export                   saves agent_decisions.json + evalu

## 7. Run the Pipeline

In [68]:
initial_state: AgentState = {
    "df":                           None,
    "train_df":                     None,
    "valid_df":                     None,
    "raw_summary":                  None,
    "cleaned_summary":              None,
    "fe_summary":                   None,
    "cleaning_decisions_raw":       None,
    "cleaning_decisions_validated": None,
    "fe_decisions_raw":             None,
    "fe_decisions_validated":       None,
    "model_config_raw":             None,
    "model_config_validated":       None,
    "evaluation_decisions":         None,
    "dense_feature_cols":           None,
    "skipped_instructions":         None,
    "model_1_results":              None,
    "model_2_results":              None,
    "model_1_best":                 None,
    "model_2_best":                 None,
    "feature_importance":           None,
    "output_json_path":             None,
    "evaluation_report_path":       None,
}

final_state = pipeline.invoke(initial_state)

cfg     = final_state["model_config_validated"]
m1_name = cfg.get("model_1", {}).get("pyspark_class", "Model1")
m2_name = cfg.get("model_2", {}).get("pyspark_class", "Model2")
pri_met = cfg.get("primary_metric", "areaUnderROC")

print("\n" + "="*60)
print("PIPELINE COMPLETE")
print("="*60)
print(f"Agent decisions:   {final_state['output_json_path']}")
print(f"Evaluation report: {final_state['evaluation_report_path']}")
print(f"{m1_name}: {pri_met}={final_state['model_1_best'].get(pri_met) if final_state['model_1_best'] else 'N/A'}")
print(f"{m2_name}: {pri_met}={final_state['model_2_best'].get(pri_met) if final_state['model_2_best'] else 'N/A'}")
print(f"Selected model: {final_state['evaluation_decisions'].get('selected_model')}")


NODE 1: LOAD & NON-AGENT PREPARATION
Dataset path: /Users/joshlim/.cache/kagglehub/datasets/najzeko/steam-reviews-2021/versions/1


Rows after non-agent preparation: 9,635,437
Columns (22): ['app_id', 'app_name', 'review_id', 'language', 'review', 'timestamp_created', 'timestamp_updated', 'recommended', 'votes_helpful', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'author_steamid', 'author_num_games_owned', 'author_num_reviews', 'author_playtime_forever', 'author_playtime_last_two_weeks', 'author_playtime_at_review', 'author_last_played']

NODE 2: RAW SUMMARY STATISTICS


Rows: 9,635,437 | Columns: 22
Class distribution: {1: 8584444, 0: 1050993}
Duplicates — exact: 54,769 | review_id: 54,769 | user+app: 55,027
Columns with nulls: ['review', 'author_playtime_forever', 'author_playtime_last_two_weeks', 'author_playtime_at_review', 'author_last_played']

NODE 3: CLEANING AGENT

CLEANING AGENT — RAW DECISIONS
The dataset contains a significant number of rows and various columns that may require cleaning to ensure the quality and reliability of the machine learning model. Observations such as missing values, duplicates, class imbalance, and outlier presence indicate the need for specific cleaning steps. Each decision is based on the statistics provided, aiming to enhance data integrity and model performance.

--- RAW DECISIONS ---
{
  "missing_values": {
    "observations": "The 'review' column has 16,750 missing values, which is about 1.73% of the total rows. Additionally, 'author_playtime_at_review' has 11,847 missing values (0.12%).",
    "action": "Remov

  [dup] Dropped exact duplicates
  [dup] Kept latest review per user+app
  [drop] Columns dropped: ['app_id', 'review_id']
  [null] drop_rows for review


  [null] impute_median=1888.0000 for author_playtime_at_review
  [outlier] dropped rows where votes_helpful > 99
  [outlier] dropped rows where author_num_games_owned > 99


...

    {'numTrees': 100, 'maxDepth': 10, 'minInstancesPerNode': 2, 'featureSubsetStrategy': 'sqrt'} -> areaUnderROC=0.7365


    {'numTrees': 200, 'maxDepth': 15, 'minInstancesPerNode': 2, 'featureSubsetStrategy': 'sqrt'} -> areaUnderROC=0.7375


    {'numTrees': 100, 'maxDepth': 10, 'minInstancesPerNode': 5, 'featureSubsetStrategy': 'log2'} -> areaUnderROC=0.7365


    {'numTrees': 150, 'maxDepth': 10, 'minInstancesPerNode': 10, 'featureSubsetStrategy': 'sqrt'} -> areaUnderROC=0.7362


    {'numTrees': 300, 'maxDepth': 5, 'minInstancesPerNode': 2, 'featureSubsetStrategy': 'sqrt'} -> areaUnderROC=0.7261

Best LogisticRegression: areaUnderROC=0.6833
Best RandomForestClassifier: areaUnderROC=0.7375
Overlapping top-10: ['is_early_access_review', 'review_length_chars', 'review_length_words', 'review_created_datetime', 'punctuation_count', 'exclamation_count', 'question_count']

NODE 14: EVALUATION AGENT

EVALUATION AGENT
The evaluation process involved comparing model performance metrics to determine the best-performing model. Feature importance was analyzed to understand the driving factors behind review sentiment, focusing on the implications of each feature in the context of Steam. The insights gained were then synthesized into practical recommendations for developers and the platform, while also acknowledging the limitations of the analysis.

--- EVALUATION SUMMARY ---
  Selected: RandomForestClassifier (areaUnderROC=0.7375)
  Runner-up: LogisticRegression (areaUnderR

## 8. Inspect All Agent Decisions

Both raw (what the domain agent decided) and validated (what the validator translated) are shown side by side.

In [69]:
sections = [
    ("CLEANING AGENT — RAW",       "cleaning_decisions_raw"),
    ("CLEANING VALIDATOR",         "cleaning_decisions_validated"),
    ("FE AGENT — RAW",             "fe_decisions_raw"),
    ("FE VALIDATOR",               "fe_decisions_validated"),
    ("MODEL CONFIG AGENT — RAW",   "model_config_raw"),
    ("MODEL CONFIG VALIDATOR",     "model_config_validated"),
    ("EVALUATION AGENT",           "evaluation_decisions"),
]
for title, key in sections:
    data = final_state[key]
    print("=" * 60)
    print(title)
    print("=" * 60)
    if "reasoning" in data:
        print("REASONING:", data["reasoning"])
        print()
    if "translation_notes" in data:
        print("TRANSLATION NOTES:", data["translation_notes"])
        print()
    print(json.dumps({k: v for k, v in data.items() if k not in ("reasoning","translation_notes")}, indent=2))
    print()

CLEANING AGENT — RAW
REASONING: The dataset contains a significant number of rows and various columns that may require cleaning to ensure the quality and reliability of the machine learning model. Observations such as missing values, duplicates, class imbalance, and outlier presence indicate the need for specific cleaning steps. Each decision is based on the statistics provided, aiming to enhance data integrity and model performance.

{
  "decisions": {
    "missing_values": {
      "observations": "The 'review' column has 16,750 missing values, which is about 1.73% of the total rows. Additionally, 'author_playtime_at_review' has 11,847 missing values (0.12%).",
      "action": "Remove rows with missing values in the 'review' column, as this is critical for the classification task. For 'author_playtime_at_review', consider imputing missing values with the median or mean.",
      "justification": "Removing rows with missing reviews ensures that all remaining data points are valid for th

## 9. Results Summary

In [70]:
cfg     = final_state["model_config_validated"]
m1_name = cfg.get("model_1", {}).get("pyspark_class", "Model1")
m2_name = cfg.get("model_2", {}).get("pyspark_class", "Model2")
pri_met = cfg.get("primary_metric", "areaUnderROC")

for name, results_key in [(m1_name, "model_1_results"), (m2_name, "model_2_results")]:
    print("=" * 60)
    print(f"{name.upper()} — ALL CONFIGS")
    print("=" * 60)
    if final_state[results_key]:
        df_res = pd.DataFrame(final_state[results_key]).sort_values(pri_met, ascending=False)
        print(df_res.to_string(index=False))
    print()

print("=" * 60)
print("FEATURE IMPORTANCE — TOP 10 DENSE FEATURES")
print("=" * 60)
fi = final_state["feature_importance"]
for mk, mn in [("model_1", m1_name), ("model_2", m2_name)]:
    print(f"\n{mn}:")
    for r in fi.get(mk, {}).get("top10_dense", []):
        if "coefficient" in r:
            print(f"  {r['feature']:<45}  coeff={r['coefficient']:+.6f}")
        else:
            print(f"  {r['feature']:<45}  importance={r['importance']:.6f}")
    s = fi.get(mk, {})
    print(f"  Dense: {s.get('dense_share')} | TF-IDF: {s.get('tfidf_share')}")

print(f"\nOverlapping top-10: {fi.get('overlapping_top10', [])}")

print()
print("=" * 60)
print("FE TRANSFORMATIONS APPLIED")
print("=" * 60)
for i, t in enumerate(final_state["fe_decisions_validated"].get("validated_transformations", []), 1):
    print(f"  {i:2d}. [{t.get('type'):<20}] -> {t.get('output', t.get('input', '?'))}")
if final_state.get("skipped_instructions"):
    print(f"\n  Skipped: {[s.get('type') for s in final_state['skipped_instructions']]}")

LOGISTICREGRESSION — ALL CONFIGS
 regParam  maxIter  elasticNetParam  areaUnderROC  areaUnderPR     f1  accuracy  weightedPrecision  weightedRecall
    100.0      100              0.0        0.6833       0.6662 0.5087    0.5619             0.6107          0.5619
     10.0      100              0.0        0.6831       0.6662 0.6055    0.6140             0.6252          0.6140
      0.1      100              0.0        0.6820       0.6653 0.6259    0.6288             0.6332          0.6288
      1.0      100              0.0        0.6818       0.6654 0.6184    0.6232             0.6301          0.6232
      1.0      100              0.0        0.6817       0.6651 0.6184    0.6232             0.6301          0.6232

RANDOMFORESTCLASSIFIER — ALL CONFIGS
 numTrees  maxDepth  minInstancesPerNode featureSubsetStrategy  areaUnderROC  areaUnderPR     f1  accuracy  weightedPrecision  weightedRecall
      200        15                    2                  sqrt        0.7375       0.7235 0.6811 

26/04/15 06:33:36 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:707)
	at org.apache.spark.storage.BlockManagerMasterE